In [1]:
### automatically refresh the buffer
%load_ext autoreload
%autoreload 2

### solve the auto-complete issue

%config Completer.use_jedi = False
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter(action='ignore', category=FutureWarning)

### lvl 2 setups (systerm)
import os
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap,LinearSegmentedColormap,BoundaryNorm
import matplotlib.dates as mdates
import geopandas as gpd
from shapely.geometry import Point
from datetime import datetime
import h5py
import numpy as np
np.set_printoptions(suppress=True)

In [2]:
ds_ca = xr.open_dataset('/data/ggong/CALIPSO/Dustprof/SGP_annual/merge_06-21_5x5.nc').sel(time=slice("2006-01-01", "2017-12-31"))
ds_cs = xr.open_dataset('/data/ggong/CloudSat/merge_nc/SGP_merge5.nc')
ds_md = xr.open_dataset('/data/ggong/MODIS/SGP_COD_merge_5x5.nc').sel(time=slice("2006-01-01", "2017-12-31"))

### Temporal Matching of CALIPSO, CloudSat, and MODIS Overpass Events

In [3]:
import numpy as np

def get_events(ds, thr=2.0):
    t = np.sort(ds.time.values.astype("datetime64[ns]"))
    dt = np.diff(t) / np.timedelta64(1, "s")
    seg, s = [], 0
    for i, ok in enumerate(dt <= thr):
        if not ok: seg.append((s, i)); s = i + 1
    seg.append((s, len(t) - 1))
    return np.array([(t[i], t[j]) for i, j in seg], dtype="datetime64[ns]")

def _dist(a0,a1,b0,b1):
    if (b1 >= a0) and (b0 <= a1): return np.timedelta64(0,"ns")
    if b1 < a0: return a0 - b1
    return b0 - a1

def count_event_matches(A, B, tol_s=120):
    tol = np.timedelta64(int(tol_s*1e9), "ns")
    A_hit = np.zeros(len(A), bool); B_hit = np.zeros(len(B), bool); pair = 0
    for i,(a0,a1) in enumerate(A):
        m = (B[:,1] >= a0 - tol) & (B[:,0] <= a1 + tol)
        for j in np.where(m)[0]:
            if _dist(a0,a1,*B[j]) <= tol: pair += 1; A_hit[i]=1; B_hit[j]=1
    return A_hit.sum(), B_hit.sum(), pair

def matched_pairs(A, B, tol_s=120):
    tol = np.timedelta64(int(tol_s*1e9), "ns"); out = []
    for i,(a0,a1) in enumerate(A):
        m = (B[:,1] >= a0 - tol) & (B[:,0] <= a1 + tol)
        if not np.any(m): continue
        js = np.where(m)[0]
        d = np.array([_dist(a0,a1,*B[j]) for j in js])
        j = js[np.argmin(d)]
        if d.min() <= tol: out.append((i, j, float(d.min()/np.timedelta64(1,"s"))))
    return out

def matched_triples(A, B, C, tol_s=120):  # A锚点：给每个A找最近B和最近C，并要求B-C也在tol内
    AB = matched_pairs(A, B, tol_s); AC = matched_pairs(A, C, tol_s)
    b_of = {i:j for i,j,_ in AB}; c_of = {i:k for i,k,_ in AC}
    tol = np.timedelta64(int(tol_s*1e9), "ns"); triples = []
    for i in set(b_of) & set(c_of):
        j, k = b_of[i], c_of[i]
        a0,a1 = A[i]; b0,b1 = B[j]; c0,c1 = C[k]
        if _dist(b0,b1,c0,c1) <= tol:
            gap = max(_dist(a0,a1,b0,b1), _dist(a0,a1,c0,c1), _dist(b0,b1,c0,c1))
            triples.append((i, j, k, float(gap/np.timedelta64(1,"s"))))
    return triples

# ===== 1) events =====
ev_ca = get_events(ds_ca, 2.0)
ev_cs = get_events(ds_cs, 2.0)
ev_md = get_events(ds_md, 2.0)

print("CA events:", len(ev_ca), "CS events:", len(ev_cs), "MD events:", len(ev_md))

# ===== 2) pairwise match stats (tol=120s) =====
for X, Y, nX, nY in [(ev_ca, ev_cs, "CA", "CS"), (ev_ca, ev_md, "CA", "MD"), (ev_cs, ev_md, "CS", "MD")]:
    xhit, yhit, npair = count_event_matches(X, Y, tol_s=120)
    print(f"{nX}-{nY}: {nX} matched {xhit}/{len(X)}, {nY} matched {yhit}/{len(Y)}, pairs={npair}")

# ===== 3) triple matches (CA as anchor) =====
triples = matched_triples(ev_ca, ev_cs, ev_md, tol_s=120)
print("Triple matches (CA-CS-MD):", len(triples))
# triples: (i_ca, j_cs, k_md, max_gap_seconds)

CA events: 1875 CS events: 1323 MD events: 1512
CA-CS: CA matched 1320/1875, CS matched 1318/1323, pairs=1320
CA-MD: CA matched 1292/1875, MD matched 1308/1512, pairs=1310
CS-MD: CS matched 743/1323, MD matched 748/1512, pairs=748
Triple matches (CA-CS-MD): 742


### dust-cloud collocation

In [4]:
import numpy as np
import xarray as xr
from scipy.spatial import cKDTree

lon180 = lambda x: ((x + 180) % 360) - 180
xyz = lambda lat,lon: np.c_[np.cos(np.deg2rad(lat))*np.cos(np.deg2rad(lon)),
                            np.cos(np.deg2rad(lat))*np.sin(np.deg2rad(lon)),
                            np.sin(np.deg2rad(lat))]

def cs_mask_bad(a):
    a = a.astype(float, copy=False)
    a[a == -99]   = np.nan
    a[a == -999]  = np.nan
    a[a == -9999] = np.nan
    return a

In [5]:
def prep_cs_global(ds_cs):
    # CloudLayerBase/Top: km -> m
    base_km = cs_mask_bad(ds_cs["CloudLayerBase"].values)
    top_km  = cs_mask_bad(ds_cs["CloudLayerTop"].values)
    base_m = base_km * 1000.0
    top_m  = top_km  * 1000.0

    valid = np.isfinite(base_m) & np.isfinite(top_m)
    cs_single = (valid.sum(axis=1) == 1)

    # single-layer cloud top (m)
    cs_top1d_m = np.nanmax(np.where(valid, top_m, np.nan), axis=1)

    # Height already m
    H_m = cs_mask_bad(ds_cs["Height"].values)

    # Temperature
    T = cs_mask_bad(ds_cs["Temperature"].values)

    cs_Ttop_K = np.full(len(cs_top1d_m), np.nan)
    good = np.isfinite(cs_top1d_m)
    if np.any(good):
        kk = np.nanargmin(np.abs(H_m[good] - cs_top1d_m[good, None]), axis=1)
        nz = H_m.shape[1]
        for ii, k0 in zip(np.where(good)[0], kk):
            kL = max(int(k0)-1, 0)
            kR = min(int(k0)+2, nz)
            cs_Ttop_K[ii] = np.nanmean(T[ii, kL:kR])

    return cs_single, cs_top1d_m, cs_Ttop_K

In [6]:
def build_ds_mcm(
    triples, ev_ca, ev_cs, ev_md,
    ds_ca, ds_cs, ds_md,
    cs_single, cs_top1d_m, cs_Ttop_K,
    SGP = (34.107322, 39.107322, -99.987643, -94.987643),
    k_cs=5, k_md=3, warm_thr_K=273.15, low_thr_m=3000.0,
    cs_mean_vars=("CloudFraction","CloudLayerBase","CloudLayerTop",
                  "Cloud_Liq_Water_Path","Liq_Geom_Mean_Radius"),
    ca_drop_vars=("Day_Night_Flag","AVD_Aerosol_Subtype","AVD_Feature_Type","Surface_Elevation",
                  "Pure_Dust_Fine_Backscatter_Coefficient_532","Pure_Dust_Coarse_Backscatter_Coefficient_532",
                  "Pure_Dust_Fine_Mass_Concentration","Pure_Dust_Coarse_Mass_Concentration",

                  "Pure_Dust_Fine_Extinction_Coefficient_532",
                  "Pure_Dust_Coarse_Extinction_Coefficient_532")
):
    ca_tall = ds_ca.time.values
    cs_tall = ds_cs.time.values

    latmin,latmax,lonmin,lonmax = SGP

    out_ca_idx = []  # global integer indices into ds_ca.time
    out_md = {"re_1621":[], "COD_1621":[], "CWP_1621":[]}
    out_cs = {f"cs_{v}_mean":[] for v in cs_mean_vars}

    def csmean(var, idx_cs_global_1d):
        a = cs_mask_bad(ds_cs[var].isel(time=idx_cs_global_1d).values.astype(float))
        return float(np.nanmean(a))

    for i_ca, j_cs, k_md_i, _ in triples:
        a0,a1 = ev_ca[i_ca]; b0,b1 = ev_cs[j_cs]; c0,c1 = ev_md[k_md_i]
        t0,t1 = min(a0,b0,c0), max(a1,b1,c1)

        # --- CA: get GLOBAL integer indices for this window (numpy indexing)
        m_ca = (ca_tall >= t0) & (ca_tall <= t1)
        idx_ca_g = np.where(m_ca)[0]
        if idx_ca_g.size == 0:
            continue

        CA = ds_ca.isel(time=idx_ca_g)
        CS = ds_cs.sel(time=slice(t0,t1))
        MD = ds_md.sel(time=slice(t0,t1))

        if CS.time.size < k_cs or CA.time.size == 0:
            continue

        # CA in box
        ca_lat = np.asarray(CA["lat"]).astype(float).ravel()
        ca_lon = lon180(np.asarray(CA["lon"]).astype(float)).ravel()
        inb = (ca_lat>=latmin)&(ca_lat<=latmax)&(ca_lon>=lonmin)&(ca_lon<=lonmax)
        if not np.any(inb):
            continue
        ca_pos = np.where(inb)[0]  # positions within CA window

        # CS KDTree
        cs_lat = np.asarray(CS["lat"]).astype(float).ravel()
        cs_lon = lon180(np.asarray(CS["lon"]).astype(float)).ravel()
        idx_cs_g = np.where((cs_tall>=t0)&(cs_tall<=t1))[0]

        nn5 = np.atleast_2d(
            cKDTree(xyz(cs_lat,cs_lon)).query(
                xyz(ca_lat[ca_pos],ca_lon[ca_pos]),
                k=k_cs
            )[1]
        )

        # MODIS KDTree
        md_lat = np.asarray(MD["lat"]).astype(float).ravel()
        md_lon = lon180(np.asarray(MD["lon"]).astype(float)).ravel()
        m = np.isfinite(md_lat)&np.isfinite(md_lon)
        if m.sum()==0:
            continue

        md_lat, md_lon = md_lat[m], md_lon[m]
        k_use = int(min(k_md, md_lat.size))
        md_nn = cKDTree(xyz(md_lat,md_lon)).query(
            xyz(ca_lat[ca_pos],ca_lon[ca_pos]),
            k=k_use
        )[1]
        md_nn = md_nn[:,None] if np.asarray(md_nn).ndim==1 else np.asarray(md_nn)

        md_re  = np.asarray(MD["re_1621"]).astype(float).ravel()[m]
        md_cod = np.asarray(MD["COD_1621"]).astype(float).ravel()[m]
        md_cwp = np.asarray(MD["CWP_1621"]).astype(float).ravel()[m]

        for r, p in enumerate(ca_pos):
            cs5 = idx_cs_g[nn5[r]].astype(int)

            if not np.all(cs_single[cs5]):
                continue

            topm = np.nanmean(cs_top1d_m[cs5])  # m
            if (not np.isfinite(topm)) or (topm>=low_thr_m):
                continue

            Tm = np.nanmean(cs_Ttop_K[cs5])
            if (not np.isfinite(Tm)) or (Tm<=warm_thr_K):
                continue

            jj = md_nn[r]
            out_md["re_1621"].append(np.nanmean(md_re[jj]))
            out_md["COD_1621"].append(np.nanmean(md_cod[jj]))
            out_md["CWP_1621"].append(np.nanmean(md_cwp[jj]))

            for v in cs_mean_vars:
                out_cs[f"cs_{v}_mean"].append(csmean(v, cs5))

            # GLOBAL integer index into ds_ca.time
            out_ca_idx.append(int(idx_ca_g[p]))

    if len(out_ca_idx) == 0:
        raise RuntimeError("No valid samples found. Check thresholds / ENA box / inputs.")

    out_ca_idx = np.asarray(out_ca_idx, dtype=int)
    t = ds_ca["time"].isel(time=out_ca_idx).values

    # CA lightweight selection (explicitly drops the 2 extinction vars)
    ca_sel = (ds_ca.isel(time=out_ca_idx)
              .drop_vars([v for v in ca_drop_vars if v in ds_ca], errors="ignore")
              .rename({"time":"sample"})
              .assign_coords(time=("sample", t))
              .assign_coords(ca_index=("sample", out_ca_idx)))

    if "lat" in ca_sel: ca_sel = ca_sel.rename({"lat":"ca_lat"})
    if "lon" in ca_sel: ca_sel = ca_sel.rename({"lon":"ca_lon"})

    cs_ds = xr.Dataset({k:(("sample",), np.asarray(v,float)) for k,v in out_cs.items()},
                       coords={"time":("sample", t)})

    md_ds = xr.Dataset({k:(("sample",), np.asarray(v,float)) for k,v in out_md.items()},
                       coords={"time":("sample", t)})

    return xr.merge([ca_sel, cs_ds, md_ds])

In [7]:
def build_ds_mca_from_indices(ds_ca, ds_mcm,
                              fine_var="Pure_Dust_Fine_Extinction_Coefficient_532",
                              coarse_var="Pure_Dust_Coarse_Extinction_Coefficient_532",
                              height_dim="height"):
    """
    Build ds_mca containing the two CA extinction variables with dims (sample, height).
    Uses numpy indexing on the first dimension of ds_ca[var].values.
    """
    idx = ds_mcm["ca_index"].values.astype(int)  # (sample,)
    t = ds_mcm["time"].values                    # (sample,)

    fine_np = ds_ca[fine_var].values[idx, :]     # (sample, height)
    coarse_np = ds_ca[coarse_var].values[idx, :] # (sample, height)

    # Get height coordinate from ds_ca if available; otherwise create an index
    if height_dim in ds_ca.dims:
        h = ds_ca[height_dim].values
        coords = {"time": ("sample", t), height_dim: h}
        dims = ("sample", height_dim)
    else:
        # fallback: infer height length from array
        h = np.arange(fine_np.shape[1])
        coords = {"time": ("sample", t), "height": h}
        dims = ("sample", "height")

    ds_mca = xr.Dataset(
        {
            fine_var: (dims, fine_np),
            coarse_var: (dims, coarse_np),
        },
        coords=coords
    )

    return ds_mca

In [8]:
# 1) precompute CS
cs_single, cs_top1d_m, cs_Ttop_K = prep_cs_global(ds_cs)

# 2) build core matched dataset (light)
ds_mcm = build_ds_mcm(
    triples, ev_ca, ev_cs, ev_md,
    ds_ca, ds_cs, ds_md,
    cs_single, cs_top1d_m, cs_Ttop_K,
    SGP = (34.107322, 39.107322, -99.987643, -94.987643),
    k_cs=5, k_md=3, warm_thr_K=273.15, low_thr_m=3000.0
)

print("ds_mcm N samples:", ds_mcm.sizes["sample"])
print("ds_mcm vars:", list(ds_mcm.data_vars))
print("ds_mcm coords:", list(ds_mcm.coords))

# 3) build ds_mca from numpy indices
ds_mca = build_ds_mca_from_indices(
    ds_ca, ds_mcm,
    fine_var="Pure_Dust_Fine_Extinction_Coefficient_532",
    coarse_var="Pure_Dust_Coarse_Extinction_Coefficient_532",
    height_dim="height"   
)

ds_mcm N samples: 3135
ds_mcm vars: ['cs_CloudFraction_mean', 'cs_CloudLayerBase_mean', 'cs_CloudLayerTop_mean', 'cs_Cloud_Liq_Water_Path_mean', 'cs_Liq_Geom_Mean_Radius_mean', 're_1621', 'COD_1621', 'CWP_1621']
ds_mcm coords: ['sample', 'ca_lat', 'ca_lon', 'height', 'time', 'ca_index']


In [16]:
ds_mcm.to_netcdf('/data/ggong/CloudSat/matched_CCM/Cloudsat_MODIS_SGP.nc')

In [17]:
ds_mca.to_netcdf('/data/ggong/CloudSat/matched_CCM/CALIPSO_SGP.nc')